# Rank-one causal adaptation: all experiments

Every experiment behind the WiML abstract, in run order. CPU end to end.

| section | question | headline |
|---|---|---|
| 1-5 | synthetic pipeline, G=50 | foundation 0.92/0.90 |
| 6 | L1 sweep | l1=0 diagnostic separates regularization from optimization |
| 7 | closed loop | active acquisition beats random |
| 8 | rank1 vs dense vs no_adapt | factorization carries the result |
| 9 | why rescue ranking fails | rho(disease, healthy rescue) = 0.967 |
| 10-11 | real Sachs + semi-synthetic control | at chance; 2 of 7 regimes carry signal |
| 12 | identifiability vs network size | lift over chance grows 6x, G=11 to 50 |
| 13 | topological-order robustness | structural metrics don't predict adapter accuracy |

**Run top to bottom in one kernel** - sections 8-13 reuse `sim`, `fusion`, `corpus`, `foundation`,
`fit_ensemble` from Part 1.

**Three bugs found along the way.** (a) Adapter L1 was ~100x too strong, driving `v` to exactly
zero. (b) The rescue task leaked: the readout loaded on genes candidate interventions clamped
directly, so an unadapted foundation scored identically. (c) On real data, standardizing single
cells *then* averaging into pseudo-replicates shrank variance by sqrt(n), leaving |W0| ~ 0.07 -
no network for the adapter to find a lesion in.

In [3]:
!pip install -q torch pandas scipy matplotlib
!pip download -q --no-deps -d /tmp/cdt cdt  # bundles the Sachs data used in section 10

## Sections 1-3 - Config, simulator, corpus, foundation

Gate 1 (edge recovery) and gate 2 (phenotype drop) are asserted. If either fires, stop.

In [4]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass, field
from scipy.stats import spearmanr

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FAST = True
G = 50 if FAST else 60
N_CONTEXTS = 10 if FAST else 12
EDGE_P, KD_LEVEL, NOISE, REPS = 0.06, -1.5, 0.10, 6
FOUND_EPOCHS = 500 if FAST else 900
ADAPT_STEPS = 300 if FAST else 700
N_MEMBERS, N_ROUNDS, ORACLE_N = 6, 8 if FAST else 14, 150 if FAST else 400
ADAPT_L1 = 1e-4
print(DEVICE, "| FAST =", FAST)

cpu | FAST = True


In [5]:
def dag_depth(W):
    n, d = W.shape[0], np.zeros(W.shape[0], int)
    for i in range(n):
        par = np.where(W[:i, i] != 0)[0]
        if len(par):
            d[i] = d[par].max() + 1
    return int(d.max())


@dataclass
class Simulator:
    n_genes: int = G
    n_contexts: int = N_CONTEXTS
    edge_p: float = EDGE_P
    noise: float = NOISE
    seed: int = 0
    W0: np.ndarray = field(default=None, repr=False)

    def __post_init__(self):
        rng = np.random.default_rng(self.seed)
        n = self.n_genes
        mask = np.tril(rng.random((n, n)) < self.edge_p, -1).T
        self.W0 = (mask * rng.normal(0, 0.9, (n, n))).astype(np.float32)
        self.b = rng.normal(0, 0.3, n).astype(np.float32)
        self.gains = np.clip(rng.normal(1.0, 0.25, (self.n_contexts, n)), 0.3, 1.8).astype(np.float32)
        self.w_y = None
        self.depth = dag_depth(self.W0) + 2

    def W_context(self, c, dW=None):
        W = self.W0 * self.gains[c][None, :]
        if dW is not None:
            W = W + dW
        return np.tril(W.T, -1).T

    def simulate(self, W, n_samples, interventions=None, rng=None):
        rng = rng or np.random.default_rng()
        n = self.n_genes
        eps = rng.normal(0, self.noise, (n_samples, n)).astype(np.float32)
        m = np.zeros(n, np.float32); vals = np.zeros(n, np.float32)
        for g, v in (interventions or {}).items():
            m[g], vals[g] = 1.0, v
        X = np.zeros((n_samples, n), np.float32)
        for _ in range(self.depth):
            X = np.tanh(X @ W + self.b) + eps
            X = X * (1 - m) + vals * m
        return X

    def phenotype(self, X):
        return 1.0 / (1.0 + np.exp(-(X @ self.w_y)))


def make_fusion(sim, n_targets=6, strength=1.6, seed=1):
    rng = np.random.default_rng(seed)
    f = int(rng.integers(2, sim.n_genes // 3))
    tgts = rng.choice(np.arange(f + 1, sim.n_genes), size=n_targets, replace=False)
    v = np.zeros(sim.n_genes, np.float32)
    v[tgts] = strength * rng.uniform(0.8, 1.2, len(tgts))
    dW = np.zeros((sim.n_genes, sim.n_genes), np.float32)
    dW[f, :] = v
    return {"anchor": f, "v": v, "dW": dW, "targets": np.sort(tgts)}


def descendants(W, f):
    seen, frontier = set(), [f]
    while frontier:
        node = frontier.pop()
        for child in np.where(W[node, :] != 0)[0]:
            if int(child) not in seen:
                seen.add(int(child)); frontier.append(int(child))
    return np.array(sorted(seen), int)


def attach_phenotype(sim, fusion, context, seed=0):
    """FIX 3. Readout loads on DEEP descendants only - the fusion's direct targets are
    excluded, so they remain legitimate upstream intervention points and no candidate
    can move the phenotype simply by clamping a gene the readout reads."""
    rng = np.random.default_rng(seed)
    W_dis = sim.W_context(context, dW=fusion["dW"])
    deep = np.setdiff1d(descendants(W_dis, fusion["anchor"]), fusion["targets"])
    if len(deep) < 3:
        raise RuntimeError("too few deep descendants; raise n_targets or resample the fusion seed")
    w = np.zeros(sim.n_genes, np.float32)
    w[deep] = rng.uniform(0.6, 1.0, len(deep))
    sim.w_y = (w / np.linalg.norm(w)).astype(np.float32)

    W_h = sim.W_context(context)
    f = lambda W, s: sim.phenotype(sim.simulate(W, 400, rng=np.random.default_rng(s))).mean()
    if f(W_dis, 11) > f(W_h, 10):
        sim.w_y = -sim.w_y
    return f(W_h, 10), f(W_dis, 11), deep


sim = Simulator()
DISEASE_CONTEXT = 3
fusion = make_fusion(sim)
y_h, y_d, deep = attach_phenotype(sim, fusion, DISEASE_CONTEXT)
W_healthy = sim.W_context(DISEASE_CONTEXT)
W_disease = sim.W_context(DISEASE_CONTEXT, dW=fusion["dW"])

SUPPORT = set(np.where(np.abs(sim.w_y) > 1e-6)[0].tolist())
CAND_GENES = [g for g in range(G) if g not in SUPPORT]
CANDIDATES = [{g: KD_LEVEL} for g in CAND_GENES]

print(f"depth={sim.depth} anchor={fusion['anchor']} targets={fusion['targets']}")
print(f"readout on {len(deep)} deep descendants | {len(CANDIDATES)} non-leaky candidates")
print(f"phenotype healthy={y_h:.3f} disease={y_d:.3f} drop={y_h - y_d:+.3f}")
assert y_h - y_d > 0.05, "phenotype drop too small - raise strength/n_targets"
assert len(CANDIDATES) > 15, "too few non-leaky candidates - shrink the readout support"
print("GATE 2 PASSED")

depth=10 anchor=8 targets=[10 14 27 36 42 45]
readout on 22 deep descendants | 28 non-leaky candidates
phenotype healthy=0.475 disease=0.420 drop=+0.055
GATE 2 PASSED


In [6]:
N_SINGLE, N_DOUBLE = (30, 10) if FAST else (40, 15)


def build_corpus(sim, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for c in range(sim.n_contexts):
        W = sim.W_context(c)
        specs = [{}] + [{int(g): KD_LEVEL} for g in rng.choice(sim.n_genes, N_SINGLE, replace=False)]
        for _ in range(N_DOUBLE):
            a, b = rng.choice(sim.n_genes, 2, replace=False)
            specs.append({int(a): KD_LEVEL, int(b): KD_LEVEL})
        rows += [{"context": c, "spec": s,
                  "X": sim.simulate(W, REPS, interventions=s, rng=rng).mean(0)} for s in specs]
    return rows


def build_drugs(sim, n_drugs=40, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for d in range(n_drugs):
        tgt = int(rng.choice(CAND_GENES))          # drugs act outside the readout too
        pool = [g for g in CAND_GENES if g != tgt]
        off = rng.choice(pool, rng.integers(0, 3), replace=False)
        out.append({"drug_id": f"CMP{d:03d}", "target": tgt,
                    "efficacy": float(rng.uniform(0.5, 1.0)),
                    "off_targets": [int(o) for o in off],
                    "off_efficacy": float(rng.uniform(0.1, 0.4))})
    return pd.DataFrame(out)


def drug_intervention(d):
    spec = {d["target"]: d["efficacy"] * KD_LEVEL}
    for o in d["off_targets"]:
        spec[o] = d["off_efficacy"] * KD_LEVEL
    return spec


corpus, drugs = build_corpus(sim), build_drugs(sim)
print(f"{len(corpus)} foundation experiments, {len(drugs)} compounds")

410 foundation experiments, 40 compounds


In [7]:
class FoundationSCM(nn.Module):
    def __init__(self, n_genes, n_contexts, depth):
        super().__init__()
        self.n, self.depth = n_genes, depth
        self.W_raw = nn.Parameter(torch.randn(n_genes, n_genes) * 0.05)
        self.b = nn.Parameter(torch.zeros(n_genes))
        self.gains = nn.Parameter(torch.ones(n_contexts, n_genes))
        self.register_buffer("tri", torch.tril(torch.ones(n_genes, n_genes), -1).T)

    def W0(self):
        return self.W_raw * self.tri

    def forward(self, gain, int_mask, int_val, dW=None, db=None):
        W0 = self.W0()
        X = torch.zeros_like(gain if gain.dim() == 3 else int_mask)
        for _ in range(self.depth):
            pre = (X @ W0) * gain + self.b
            if dW is not None:
                D = dW * (self.tri if dW.dim() == 2 else self.tri.unsqueeze(0))
                pre = pre + (X @ D if dW.dim() == 2 else torch.einsum("ebg,egh->ebh", X, D))
            if db is not None:
                pre = pre + (db if db.dim() == 1 else db.unsqueeze(1))
            X = torch.tanh(pre) * (1 - int_mask) + int_val * int_mask
        return X


def encode(specs, n_genes):
    m = np.zeros((len(specs), n_genes), np.float32)
    v = np.zeros((len(specs), n_genes), np.float32)
    for r, s in enumerate(specs):
        for g, val in s.items():
            m[r, g], v[r, g] = 1.0, val
    return torch.tensor(m, device=DEVICE), torch.tensor(v, device=DEVICE)


def fit_foundation(model, corpus, epochs=FOUND_EPOCHS, lr=5e-2, l1=1e-3, verbose=True):
    ctx = torch.tensor([r["context"] for r in corpus], device=DEVICE)
    im, iv = encode([r["spec"] for r in corpus], model.n)
    Y = torch.tensor(np.stack([r["X"] for r in corpus]), device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        pred = model(model.gains[ctx], im, iv)
        # FIX 2: sum-over-genes data term against sum-over-entries penalty, as in v1
        loss = nn.functional.mse_loss(pred, Y) * model.n + l1 * model.W0().abs().sum()
        opt.zero_grad(); loss.backward(); opt.step()
        if verbose and ep % 150 == 0:
            print(f"  epoch {ep:4d}  loss {loss.item():.5f}")
    return model


foundation = fit_foundation(FoundationSCM(G, N_CONTEXTS, sim.depth).to(DEVICE), corpus)
with torch.no_grad():
    W_hat = foundation.W0().cpu().numpy()
ts, hs = sim.W0 != 0, np.abs(W_hat) > 0.1
tp = (ts & hs).sum()
prec, rec = tp / max(hs.sum(), 1), tp / ts.sum()
print(f"GATE 1 - edge recovery: precision {prec:.2f}  recall {rec:.2f}")
assert prec > 0.8 and rec > 0.8, "foundation prior too weak - tune l1 before continuing"

  epoch    0  loss 6.81868
  epoch  150  loss 0.23710
  epoch  300  loss 0.23365
  epoch  450  loss 0.25904
GATE 1 - edge recovery: precision 0.92  recall 0.91


In [8]:
# Snapshot Part 1 state under protected names.
# Later sections define local helpers; this guarantees they cannot shadow what
# sections 12-13 need.
SIM, FUSION, CORPUS = sim, fusion, corpus
W_DIS, GVAL, NCTX = W_disease, G, N_CONTEXTS
print("snapshot ok:", type(SIM).__name__, "| G =", GVAL, "| anchor =", FUSION["anchor"])

snapshot ok: Simulator | G = 50 | anchor = 8


## Sections 4-5 - Adapter ensemble, counterfactual engine, oracle

In [9]:
class AdapterEnsemble(nn.Module):
    def __init__(self, E, n, tau0=2.0, tau1=0.2, rank1=True, use_bias=True):
        super().__init__()
        self.E, self.n, self.rank1, self.use_bias = E, n, rank1, use_bias
        self.tau0, self.tau1 = tau0, tau1
        if rank1:
            self.u_logits = nn.Parameter(torch.randn(E, n) * 0.1)
            self.v = nn.Parameter(torch.zeros(E, n))
            self.s = nn.Parameter(torch.zeros(E, n))
        else:
            self.D = nn.Parameter(torch.zeros(E, n, n))
            self.s = nn.Parameter(torch.zeros(E, n))

    def _u(self, frac):
        tau = self.tau0 * (self.tau1 / self.tau0) ** frac
        return torch.softmax(self.u_logits / tau, dim=1)

    def dW(self, frac=1.0):
        return torch.einsum("eg,eh->egh", self._u(frac), self.v) if self.rank1 else self.D

    def db(self, frac=1.0):
        if not self.use_bias:
            return None
        return self._u(frac) * self.s if self.rank1 else self.s

    def penalty(self):
        main = self.v.abs().sum(1) if self.rank1 else self.D.abs().sum((1, 2))
        return (main + self.s.abs().sum(1)).mean()

    def anchors(self):
        w = self.u_logits if self.rank1 else self.D.abs().sum(2) + self.s.abs()
        return w.argmax(1).cpu().numpy()

    def v_mean(self):
        v = self.v if self.rank1 else self.D.sum(1)
        return v.detach().mean(0).cpu().numpy()


def fit_ensemble(foundation, context_gain, anchors, n_members=N_MEMBERS, steps=ADAPT_STEPS,
                 lr=5e-2, l1=ADAPT_L1, rank1=True, use_bias=True, seed=0):
    """context_gain: (n,) tensor. anchors: list of {'spec','X'} in the target context."""
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    E, N = n_members, len(anchors)
    idx = torch.stack([torch.arange(N)] + [torch.randint(N, (N,), generator=g)
                                           for _ in range(E - 1)]).to(DEVICE)
    im, iv = encode([a["spec"] for a in anchors], foundation.n)
    Y = torch.tensor(np.stack([a["X"] for a in anchors]), device=DEVICE)
    imE, ivE, YE = im[idx], iv[idx], Y[idx]
    gain = context_gain.detach()[None, None, :].expand(E, N, foundation.n)

    ens = AdapterEnsemble(E, foundation.n, rank1=rank1, use_bias=use_bias).to(DEVICE)
    for p in foundation.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam(ens.parameters(), lr=lr)
    for s in range(steps):
        f = s / steps
        pred = foundation(gain, imE, ivE, dW=ens.dW(f), db=ens.db(f))
        loss = nn.functional.mse_loss(pred, YE) * foundation.n + l1 * ens.penalty()
        opt.zero_grad(); loss.backward(); opt.step()
    return ens.eval()


GAIN = foundation.gains[DISEASE_CONTEXT]

In [10]:
_ORACLE_CACHE = {}


def ensemble_predict(foundation, gain_vec, ens, specs):
    im, iv = encode(specs, foundation.n)
    E, S = ens.E, len(specs)
    gain = gain_vec.detach()[None, None, :].expand(E, S, foundation.n)
    with torch.no_grad():
        return foundation(gain, im[None].expand(E, S, -1), iv[None].expand(E, S, -1),
                          dW=ens.dW(1.0), db=ens.db(1.0))


def rescue_scores(foundation, gain_vec, ens, candidates):
    wy = torch.tensor(sim.w_y, device=DEVICE)
    y0 = torch.sigmoid(ensemble_predict(foundation, gain_vec, ens, [{}]) @ wy)
    y = torch.sigmoid(ensemble_predict(foundation, gain_vec, ens, candidates) @ wy)
    d = (y - y0).cpu().numpy()
    return d.mean(0), d.std(0)


def oracle_rescue(sim, W_dis, candidates, key, n=ORACLE_N, seed=0):
    if key in _ORACLE_CACHE:
        return _ORACLE_CACHE[key]
    rng = np.random.default_rng(seed)
    y0 = sim.phenotype(sim.simulate(W_dis, n, rng=rng)).mean()
    out = np.array([sim.phenotype(sim.simulate(W_dis, n, interventions=c, rng=rng)).mean() - y0
                    for c in candidates])
    _ORACLE_CACHE[key] = out
    return out


def evaluate(foundation, gain_vec, ens, fusion, oracle):
    v_hat = ens.v_mean()
    hs, ts_ = np.abs(v_hat) > 0.2, fusion["v"] != 0
    tp = (hs & ts_).sum()
    p, r = tp / max(hs.sum(), 1), tp / ts_.sum()
    mu, sd = rescue_scores(foundation, gain_vec, ens, CANDIDATES)
    return {"anchor_hit": float((ens.anchors() == fusion["anchor"]).mean()),
            "edge_f1": float(2 * p * r / (p + r)) if p + r > 0 else 0.0,
            "rescue_rho": float(spearmanr(mu, oracle).statistic),
            "top1_true_rank": int((oracle > oracle[int(mu.argmax())]).sum()),
            "mean_epistemic": float(sd.mean())}

## Section 6 - Smoke test and L1 sweep

Fit at `l1=0` first. If that recovers the targets, only the penalty was wrong; if not, the problem
is optimization and no L1 value helps. Pick the largest `l1` that keeps recall, set `ADAPT_L1`,
re-run the config cell.

In [11]:
def run_experiment(sim, W_dis, spec, reps=REPS, rng=None):
    rng = rng or np.random.default_rng()
    return {"spec": spec, "X": sim.simulate(W_dis, reps, interventions=spec, rng=rng).mean(0)}


rng = np.random.default_rng(0)
specs = [{}] + [{int(g): KD_LEVEL} for g in rng.choice(G, 19, replace=False)]
anc = [run_experiment(sim, W_disease, s, rng=rng) for s in specs]

t0 = time.time()
rows = []
for l1 in [0.0, 1e-5, 1e-4, 1e-3, 1e-2]:
    e = fit_ensemble(foundation, GAIN, anc, l1=l1)
    found = np.where(np.abs(e.v_mean()) > 0.2)[0]
    rows.append({"l1": l1, "anchor_hit": float((e.anchors() == fusion["anchor"]).mean()),
                 "n_found": len(found),
                 "recall": len(np.intersect1d(found, fusion["targets"])) / len(fusion["targets"]),
                 "epistemic": float(rescue_scores(foundation, GAIN, e, CANDIDATES)[1].mean())})
print(f"5 fits in {time.time() - t0:.0f}s | true anchor {fusion['anchor']}, targets {fusion['targets']}")
pd.DataFrame(rows)

5 fits in 12s | true anchor 8, targets [10 14 27 36 42 45]


,l1,anchor_hit,n_found,recall,epistemic
0,0.00000,0.0,33,0.833333,0.010206
1,0.00001,0.0,35,0.833333,0.009806
2,0.00010,0.0,32,0.833333,0.009910
3,0.00100,0.0,13,0.833333,0.004783
4,0.01000,0.0,5,0.666667,0.001810


In [12]:
print(pd.DataFrame(rows).to_string(index=False))
print("true anchor", fusion["anchor"], "| true targets", fusion["targets"])

     l1  anchor_hit  n_found   recall  epistemic
0.00000         0.0       33 0.833333   0.010206
0.00001         0.0       35 0.833333   0.009806
0.00010         0.0       32 0.833333   0.009910
0.00100         0.0       13 0.833333   0.004783
0.01000         0.0        5 0.666667   0.001810
true anchor 8 | true targets [10 14 27 36 42 45]


## Section 7 - Closed loop: active vs random acquisition

In [13]:
def acquire(foundation, gain_vec, ens, pool, strategy, rng):
    if strategy == "random":
        return int(rng.integers(len(pool)))
    return int(ensemble_predict(foundation, gain_vec, ens, pool).std(0).mean(1).argmax().item())


def closed_loop(strategy="active", n_init=8, n_rounds=N_ROUNDS, seed=0):
    rng = np.random.default_rng(seed)
    oracle = oracle_rescue(sim, W_disease, CANDIDATES, key=("loop", seed), seed=seed)
    pool = [{}] + [{g: KD_LEVEL} for g in range(G)]
    init = [0] + list(rng.choice(np.arange(1, len(pool)), n_init - 1, replace=False))
    anchors = [run_experiment(sim, W_disease, pool[i], rng=rng) for i in init]
    pool = [p for i, p in enumerate(pool) if i not in init]
    log = []
    for rnd in range(n_rounds):
        ens = fit_ensemble(foundation, GAIN, anchors, seed=rnd)
        log.append({"round": rnd, "n_anchors": len(anchors), "strategy": strategy,
                    **evaluate(foundation, GAIN, ens, fusion, oracle)})
        if not pool:
            break
        anchors.append(run_experiment(sim, W_disease,
                                      pool.pop(acquire(foundation, GAIN, ens, pool, strategy, rng)),
                                      rng=rng))
    return pd.DataFrame(log), ens, oracle


t0 = time.time()
log_active, ens_active, oracle = closed_loop("active")
log_random, _, _ = closed_loop("random")
print(f"both loops: {time.time() - t0:.0f}s")
log_active
print("\n===== L1 SWEEP =====")
print(pd.DataFrame(rows).to_string(index=False))
print("\n===== CLOSED LOOP: ACTIVE =====")
print(log_active.round(3).to_string(index=False))
print("\n===== CLOSED LOOP: RANDOM =====")
print(log_random.round(3).to_string(index=False))

both loops: 35s

===== L1 SWEEP =====
     l1  anchor_hit  n_found   recall  epistemic
0.00000         0.0       33 0.833333   0.010206
0.00001         0.0       35 0.833333   0.009806
0.00010         0.0       32 0.833333   0.009910
0.00100         0.0       13 0.833333   0.004783
0.01000         0.0        5 0.666667   0.001810

===== CLOSED LOOP: ACTIVE =====
 round  n_anchors strategy  anchor_hit  edge_f1  rescue_rho  top1_true_rank  mean_epistemic
     0          8   active       0.667    0.375       0.900               1           0.019
     1          9   active       1.000    0.500       0.920               0           0.007
     2         10   active       1.000    0.522       0.964               0           0.007
     3         11   active       1.000    0.545       0.946               0           0.006
     4         12   active       1.000    0.571       0.916               0           0.009
     5         13   active       0.833    0.462       0.955               0        

In [14]:
print(log_active.round(3).to_string(index=False))
print()
print(log_random.round(3).to_string(index=False))

 round  n_anchors strategy  anchor_hit  edge_f1  rescue_rho  top1_true_rank  mean_epistemic
     0          8   active       0.667    0.375       0.900               1           0.019
     1          9   active       1.000    0.500       0.920               0           0.007
     2         10   active       1.000    0.522       0.964               0           0.007
     3         11   active       1.000    0.545       0.946               0           0.006
     4         12   active       1.000    0.571       0.916               0           0.009
     5         13   active       0.833    0.462       0.955               0           0.015
     6         14   active       0.833    0.545       0.956               0           0.007
     7         15   active       0.833    0.480       0.916               0           0.016

 round  n_anchors strategy  anchor_hit  edge_f1  rescue_rho  top1_true_rank  mean_epistemic
     0          8   random       0.667    0.375       0.900               1    

## Section 8 - The decisive comparison

`rank1` must separate from `no_adapt` on `rescue_rho`. It does not - section 9 explains why.

In [15]:
class NullEnsemble:
    E = 1
    def dW(self, frac=1.0): return None
    def db(self, frac=1.0): return None
    def anchors(self): return np.array([-1])
    def v_mean(self): return np.zeros(G, np.float32)


def sweep(k_values=(5, 10, 20, 40), seeds=(0, 1, 2)):
    oracle = oracle_rescue(sim, W_disease, CANDIDATES, key=("sweep",))
    rows = []
    for seed in seeds:
        rng = np.random.default_rng(100 + seed)
        pool = [{}] + [{g: KD_LEVEL} for g in range(G)]
        for k in k_values:
            idx = [0] + list(rng.choice(np.arange(1, len(pool)), k - 1, replace=False))
            a = [run_experiment(sim, W_disease, pool[i], rng=rng) for i in idx]
            for name, e in [("rank1", fit_ensemble(foundation, GAIN, a, seed=seed)),
                            ("dense", fit_ensemble(foundation, GAIN, a, rank1=False,
                                                   n_members=3, seed=seed)),
                            ("no_adapt", NullEnsemble())]:
                rows.append({"method": name, "k": k, "seed": seed,
                             **evaluate(foundation, GAIN, e, fusion, oracle)})
    return pd.DataFrame(rows)


t0 = time.time()
sw = sweep()
print(f"sweep: {time.time() - t0:.0f}s")
sw.groupby(["method", "k"])[["anchor_hit", "edge_f1", "rescue_rho"]].mean().round(3)

sweep: 51s


anchor_hit  edge_f1  rescue_rho
method   k                                  
dense    5        0.111    0.223       0.443
         10       0.111    0.251       0.408
         20       0.111    0.227       0.493
         40       0.111    0.269       0.462
no_adapt 5        0.000    0.000       0.938
         10       0.000    0.000       0.938
         20       0.000    0.000       0.938
         40       0.000    0.000       0.938
rank1    5        0.000    0.290       0.930
         10       0.222    0.362       0.936
         20       0.667    0.483       0.958
         40       0.722    0.520       0.961

In [16]:
print(sw.groupby(["method","k"])[["anchor_hit","edge_f1","rescue_rho"]].mean().round(3).to_string())

             anchor_hit  edge_f1  rescue_rho
method   k                                  
dense    5        0.111    0.223       0.443
         10       0.111    0.251       0.408
         20       0.111    0.227       0.493
         40       0.111    0.269       0.462
no_adapt 5        0.000    0.000       0.938
         10       0.000    0.000       0.938
         20       0.000    0.000       0.938
         40       0.000    0.000       0.938
rank1    5        0.000    0.290       0.930
         10       0.222    0.362       0.936
         20       0.667    0.483       0.958
         40       0.722    0.520       0.961


## Section 9 - Why rescue ranking cannot discriminate

In [17]:
# Why no_adapt is strong: does the lesion change WHICH intervention helps?
o_dis = oracle_rescue(sim, W_disease, CANDIDATES, key=("d",))
o_hea = oracle_rescue(sim, sim.W_context(DISEASE_CONTEXT), CANDIDATES, key=("h",))
print("rho(disease-rescue, healthy-rescue) =", round(float(spearmanr(o_dis, o_hea).statistic), 3))
print("-> near 1.0 means base topology decides the ranking; no_adapt gets it for free")


rho(disease-rescue, healthy-rescue) = 0.967
-> near 1.0 means base topology decides the ranking; no_adapt gets it for free


## Part 2 - Real data (Sachs et al. 2005)

11 phospho-proteins, 7,466 single cells, nine reagent conditions, each inhibiting a **known**
protein - so hiding that identity gives real ground truth for anchor recovery.

The pooled matrix ships inside the `cdt` wheel. It has no regime labels, but block sizes sum to
exactly 7,466 and both published landmarks - the 1,755 observational subset and the 5,846 DCDI
interventional split - fall on block boundaries 2 and 7, so labels are recoverable by row index.

In [18]:
import zipfile, io, glob
z = zipfile.ZipFile(glob.glob('/tmp/cdt/*.whl')[0])
sachs_data = pd.read_csv(io.BytesIO(z.read('cdt/data/resources/cyto_full_data.csv')))
sachs_target = pd.read_csv(io.BytesIO(z.read('cdt/data/resources/cyto_full_target.csv')))
sachs_data.to_csv('sachs_data.csv', index=False)
print(sachs_data.shape, sachs_data.columns.tolist())
print(sachs_target)   # contains plcg->PIP2->PIP3->plcg, a 3-cycle; broken canonically below

sizes = [853,902,911,723,810,799,848,913,707]
print("sum:", sum(sizes), "| rows:", len(sachs_data))
print("first two blocks:", sum(sizes[:2]), "= the 1,755 observational subset")
print("first seven blocks:", sum(sizes[:7]), "= the 5,846 DCDI interventional split")

(7466, 11) ['praf', 'pmek', 'plcg', 'PIP2', 'PIP3', 'p44/42', 'pakts473', 'PKA', 'PKC', 'P38', 'pjnk']
   Cause    Effect
0   PIP2       PKC
1   plcg      PIP2
2   PIP3      plcg
3   PIP2      PIP3
4   plcg       PKC
5    PKC      pjnk
6    PKC       P38
7    PKA       P38
8    PKC      praf
9    PKA      praf
10  praf      pmek
11  pmek    p44/42
12   PKC      pmek
13   PKA  pakts473
14  PIP3  pakts473
15   PKA      pmek
16   PKA    p44/42
17   PKA      pjnk
sum: 7466 | rows: 7466
first two blocks: 1755 = the 1,755 observational subset
first seven blocks: 5846 = the 5,846 DCDI interventional split


## Sections 10-11 - Real leave-one-regime-out, and semi-synthetic control

The semi-synthetic control makes the real result interpretable: injecting a *known* rank-one
lesion into the Sachs-fitted network separates "the adapter is wrong" from "the foundation is wrong".

In [19]:
import numpy as np, pandas as pd, torch, time

PROT=['plcg','PIP3','PIP2','PKC','PKA','praf','pmek','p44/42','pakts473','P38','pjnk']
EDGES=[('plcg','PIP2'),('plcg','PIP3'),('PIP3','PIP2'),('PIP2','PKC'),('plcg','PKC'),
       ('PKC','pjnk'),('PKC','P38'),('PKA','P38'),('PKC','praf'),('PKA','praf'),
       ('praf','pmek'),('pmek','p44/42'),('PKC','pmek'),('PKA','pakts473'),
       ('PIP3','pakts473'),('PKA','pmek'),('PKA','p44/42'),('PKA','pjnk')]
BLOCKS=[('cd3cd28',853,None),('cd3cd28icam2',902,None),('cd3cd28+aktinhib',911,'pakts473'),
        ('cd3cd28+g0076',723,'PKC'),('cd3cd28+psitect',810,'PIP2'),('cd3cd28+u0126',799,'pmek'),
        ('cd3cd28+ly',848,'PIP3'),('pma',913,'PKC'),('b2camp',707,'PKA')]
N=11; idx={p:i for i,p in enumerate(PROT)}
A=np.zeros((N,N),np.float32)
for a,b in EDGES: A[idx[a],idx[b]]=1.0
DEPTH=dag_depth(A)+2
TGT={b[0]:b[2] for b in BLOCKS}

raw=pd.read_csv('sachs_data.csv')
L=np.log1p(raw.clip(lower=0))[PROT].to_numpy(np.float32)
L=(L-L.mean(0))/(L.std(0)+1e-8)                       # standardize single cells, no averaging
o=np.cumsum([0]+[b[1] for b in BLOCKS])
cells={b[0]:L[o[i]:o[i+1]] for i,b in enumerate(BLOCKS)}
TRI=np.tril(np.ones((N,N)),-1).T
NCELL=400

def build(sub, use_int=True, seed=0):
    ctx={r:i for i,r in enumerate(sub)}; cp=[]
    for r in sub:
        X=cells[r]; sel=np.random.default_rng(seed).permutation(len(X))[:NCELL]
        spec={idx[TGT[r]]: float(np.median(X[:,idx[TGT[r]]]))} if (use_int and TGT[r]) else {}
        cp+=[{'context':ctx[r],'spec':spec,'X':x} for x in X[sel]]
    return cp, ctx

def fit(sub, l1=1e-4, epochs=1500):
    cp,ctx=build(sub)
    f=fit_foundation(FoundationSCM(N,len(sub),DEPTH).to(DEVICE),cp,epochs=epochs,l1=l1,verbose=False)
    with torch.no_grad():
        W=f.W0().cpu().numpy(); hs=np.abs(W)>0.1; tp=(hs&(A>0)).sum()
    return f,(tp/max(hs.sum(),1),tp/A.sum(),float(np.abs(W[W!=0]).mean()))

full,st=fit(list(cells))
print(f'foundation (all regimes): prec {st[0]:.2f} rec {st[1]:.2f} |W0| {st[2]:.3f}')
with torch.no_grad():
    W0=full.W0().cpu().numpy(); B0=full.b.detach().cpu().numpy()
    GA=full.gains.detach().mean(0); Gn=GA.cpu().numpy()

def simulate_network(W,n,noise=0.25,rng=None):
    rng=rng or np.random.default_rng(0)
    eps=rng.normal(0,noise,(n,N)).astype(np.float32); X=np.zeros((n,N),np.float32)
    for _ in range(DEPTH): X=np.tanh((X@W)*Gn+B0)+eps
    return X

def lesion(mode,anchor,strength=0.9,n_t=4,seed=0):
    r=np.random.default_rng(seed); dW=np.zeros((N,N),np.float32)
    cand=np.arange(anchor+1,N) if mode=='row' else np.arange(0,anchor)
    if len(cand)==0: return None
    t=r.choice(cand,min(n_t,len(cand)),replace=False)
    if mode=='row': dW[anchor,t]=strength*r.uniform(.8,1.2,len(t))
    else:           dW[t,anchor]=strength*r.uniform(.8,1.2,len(t))
    return (dW*TRI).astype(np.float32)

def probe(ml,am,l1,k,anchor,seed):
    dW=lesion(ml,anchor,seed=seed)
    if dW is None: return None
    X=simulate_network(W0+dW,max(k,50),rng=np.random.default_rng(seed))
    anc=[{'spec':{},'X':x} for x in X[:k]]
    e=fit_ensemble(full,GA,anc,l1=l1,seed=seed,rank1=(am!='dense'),n_members=6 if am!='dense' else 3)
    if am=='col':
        with torch.no_grad(): D=e.dW(1.0).mean(0).cpu().numpy()
        return float(int(np.abs(D).sum(0).argmax())==anchor)
    return float((e.anchors()==anchor).mean())

print('\n=== SEMI-SYNTHETIC on corrected foundation: L1 sweep (row lesion/adapter, k=20) ===')
rr=[]
for l1 in [0.0,1e-4,1e-3,1e-2,5e-2]:
    h=[probe('row','row',l1,20,a,s) for a in [2,3,4,5] for s in range(2)]
    h=[x for x in h if x is not None]; rr.append({'l1':l1,'hit':round(float(np.mean(h)),3)})
sw=pd.DataFrame(rr); print(sw.to_string(index=False))
BEST=float(sw.loc[sw.hit.idxmax(),'l1']); print(f'best l1={BEST} | chance={1/N:.3f}')

print('\n=== lesion type vs adapter mode (k=20) ===')
res=[]
for ml in ['row','col']:
    for am in ['row','col','dense']:
        h=[probe(ml,am,BEST,20,a,s) for a in [2,3,4,5,6] for s in range(3)]
        h=[x for x in h if x is not None]
        res.append({'lesion':ml,'adapter':am,'hit':round(float(np.mean(h)),3)})
print(pd.DataFrame(res).to_string(index=False))

print('\n=== sample efficiency (row lesion/adapter) ===')
print(pd.DataFrame([{'k':k,'hit':round(float(np.mean([x for x in \
    [probe('row','row',BEST,k,a,s) for a in [2,3,4,5,6] for s in range(3)] if x is not None])),3)} \
    for k in [3,5,10,20,40]]).to_string(index=False))

print('\n=== REAL leave-one-regime-out anchor recovery ===')
rows=[]
for regime,tgt in TGT.items():
    if tgt is None: continue
    ti=idx[tgt]; sub=[r for r in cells if r!=regime]
    f,s2=fit(sub); g=f.gains.detach().mean(0)
    base=np.stack([c['X'] for c in build(sub)[0]]).mean(0)
    for seed in range(3):
        X=cells[regime]; r=np.random.default_rng(seed)
        sel=X[r.permutation(len(X))]
        for k in [5,20,50]:
            anc=[{'spec':{},'X':x} for x in sel[:k]]
            sh=np.abs(np.stack([a['X'] for a in anc]).mean(0)-base)
            rows.append({'regime':regime,'method':'max_shift','k':k,'hit':float(int(sh.argmax())==ti)})
            for nm,kw in [('rank1',{}),('dense',{'rank1':False,'n_members':3})]:
                e=fit_ensemble(f,g,anc,l1=BEST,seed=seed,**kw)
                rows.append({'regime':regime,'method':nm,'k':k,'hit':float((e.anchors()==ti).mean())})
    print(f'  {regime:20s} prec {s2[0]:.2f} rec {s2[1]:.2f} |W0| {s2[2]:.2f}')
R=pd.DataFrame(rows)
print(f'\nchance = {1/N:.3f}')
print(R.groupby(['method','k']).hit.mean().round(3).unstack().to_string())
print('\nper regime (rank1):')
print(R[R.method=='rank1'].pivot_table(index='regime',columns='k',values='hit').round(2).to_string())

foundation (all regimes): prec 0.46 rec 0.89 |W0| 0.681

=== SEMI-SYNTHETIC on corrected foundation: L1 sweep (row lesion/adapter, k=20) ===
    l1   hit
0.0000 0.250
0.0001 0.250
0.0010 0.250
0.0100 0.229
0.0500 0.292
best l1=0.05 | chance=0.091

=== lesion type vs adapter mode (k=20) ===
lesion adapter   hit
   row     row 0.178
   row     col 0.067
   row   dense 0.178
   col     row 0.144
   col     col 0.267
   col   dense 0.156

=== sample efficiency (row lesion/adapter) ===
 k   hit
 3 0.178
 5 0.211
10 0.178
20 0.178
40 0.156

=== REAL leave-one-regime-out anchor recovery ===
  cd3cd28+aktinhib     prec 0.41 rec 0.83 |W0| 0.55
  cd3cd28+g0076        prec 0.43 rec 0.72 |W0| 0.28
  cd3cd28+psitect      prec 0.50 rec 0.89 |W0| 0.46
  cd3cd28+u0126        prec 0.43 rec 0.83 |W0| 0.57
  cd3cd28+ly           prec 0.46 rec 0.89 |W0| 0.53
  pma                  prec 0.46 rec 0.94 |W0| 0.58
  b2camp               prec 0.42 rec 0.89 |W0| 0.63

chance = 0.091
k             5      20     5

## Section 12 - Identifiability vs network size

In [20]:
import numpy as np, pandas as pd, torch, time

# ---- synthetic SCM generator at arbitrary N, matched to real Sachs weight scale
def make_net(N, edge_p=0.10, w_scale=0.6, seed=0):
    r = np.random.default_rng(seed)
    tri = np.tril(np.ones((N, N)), -1).T
    W = ((r.random((N, N)) < edge_p) * r.normal(0, w_scale, (N, N)) * tri).astype(np.float32)
    b = r.normal(0, 0.3, N).astype(np.float32)
    g = np.clip(r.normal(1.0, 0.2, N), 0.4, 1.6).astype(np.float32)
    d = 0
    dep = np.zeros(N, int)
    for i in range(N):
        p = np.where(W[:i, i] != 0)[0]
        if len(p): dep[i] = dep[p].max() + 1
    return W, b, g, tri, int(dep.max()) + 2

def simulate_network(W, b, g, depth, n, noise=0.25, rng=None):
    rng = rng or np.random.default_rng(0)
    eps = rng.normal(0, noise, (n, W.shape[0])).astype(np.float32)
    X = np.zeros((n, W.shape[0]), np.float32)
    for _ in range(depth):
        X = np.tanh((X @ W) * g + b) + eps
    return X

def lesion(N, tri, mode, anchor, strength=0.9, n_t=4, seed=0):
    r = np.random.default_rng(seed)
    dW = np.zeros((N, N), np.float32)
    cand = np.arange(anchor + 1, N) if mode == 'row' else np.arange(0, anchor)
    if len(cand) == 0: return None
    t = r.choice(cand, min(n_t, len(cand)), replace=False)
    if mode == 'row': dW[anchor, t] = strength * r.uniform(.8, 1.2, len(t))
    else:             dW[t, anchor] = strength * r.uniform(.8, 1.2, len(t))
    return (dW * tri).astype(np.float32)

def anchor_from(ens, mode, N):
    """FIX: col mode reads the column with the largest inferred change.
    The previous version averaged dW over members before taking argmax, which
    cancels sign-opposed members; take |.| per member, then vote."""
    with torch.no_grad():
        D = ens.dW(1.0).detach().cpu().numpy()          # (E, N, N)
    if mode == 'col':
        votes = np.abs(D).sum(1).argmax(1)              # sum over rows -> per-column mass
    elif mode == 'dense':
        votes = np.abs(D).sum(2).argmax(1)
    else:
        return ens.anchors()
    return votes

def build_foundation(N, W, b, g, depth, n_ctx=6, n_per=250, seed=0):
    """Foundation sees the healthy network across several contexts, with interventions."""
    r = np.random.default_rng(seed)
    corpus = []
    for c in range(n_ctx):
        gc = np.clip(g * r.normal(1.0, 0.12, N), 0.3, 1.8).astype(np.float32)
        tg = r.choice(N, 6, replace=False)
        for j in range(n_per):
            spec = {int(tg[j % 6]): -1.2} if j % 3 == 0 else {}
            m = np.zeros(N, np.float32); v = np.zeros(N, np.float32)
            for gi, val in spec.items(): m[gi], v[gi] = 1, val
            X = np.zeros((1, N), np.float32)
            eps = r.normal(0, 0.25, (1, N)).astype(np.float32)
            for _ in range(depth):
                X = np.tanh((X @ W) * gc + b) + eps
                X = X * (1 - m) + v * m
            corpus.append({'context': c, 'spec': spec, 'X': X[0]})
    f = fit_foundation(FoundationSCM(N, n_ctx, depth).to(DEVICE), corpus,
                       epochs=1200, l1=1e-4, verbose=False)
    with torch.no_grad():
        Wh = f.W0().cpu().numpy()
    hs = np.abs(Wh) > 0.1; tp = (hs & (W != 0)).sum()
    return f, (tp / max(hs.sum(), 1), tp / max((W != 0).sum(), 1))

L1 = 1e-3
rows = []
t0 = time.time()
for N in [11, 20, 35, 50]:
    W, b, g, tri, depth = make_net(N, seed=1)
    f, st = build_foundation(N, W, b, g, depth)
    GA = f.gains.detach().mean(0)
    print(f'N={N:3d}  foundation prec {st[0]:.2f} rec {st[1]:.2f}  depth {depth}', flush=True)
    anchors = list(range(2, min(N // 2 + 2, 8)))
    for mode in ['row', 'col']:
        for k in [10, 40]:
            hits = []
            for a in anchors:
                for s in range(3):
                    dW = lesion(N, tri, mode, a, seed=s)
                    if dW is None: continue
                    X = simulate_network(W + dW, b, g, depth, max(k, 60), rng=np.random.default_rng(s))
                    anc = [{'spec': {}, 'X': x} for x in X[:k]]
                    e = fit_ensemble(f, GA, anc, l1=L1, seed=s)
                    hits.append(float((anchor_from(e, mode, N) == a).mean()))
            rows.append({'N': N, 'lesion': mode, 'k': k,
                         'hit': round(float(np.mean(hits)), 3),
                         'chance': round(1 / N, 3),
                         'lift': round(float(np.mean(hits)) * N, 2)})
    print(pd.DataFrame([r for r in rows if r['N'] == N]).to_string(index=False), flush=True)

print(f'\ntotal {time.time()-t0:.0f}s')
df = pd.DataFrame(rows)
df.to_csv('nsweep.csv', index=False)
print('\n=== anchor recovery vs network size ===')
print(df.pivot_table(index=['lesion', 'k'], columns='N', values='hit').to_string())
print('\n=== lift over chance (1.0 = chance) ===')
print(df.pivot_table(index=['lesion', 'k'], columns='N', values='lift').to_string())

N= 11  foundation prec 0.29 rec 0.40  depth 4
 N lesion  k   hit  chance  lift
11    row 10 0.156   0.091  1.71
11    row 40 0.111   0.091  1.22
11    col 10 0.433   0.091  4.77
11    col 40 0.444   0.091  4.89
N= 20  foundation prec 0.18 rec 0.32  depth 5
 N lesion  k   hit  chance  lift
20    row 10 0.148    0.05  2.96
20    row 40 0.120    0.05  2.41
20    col 10 0.389    0.05  7.78
20    col 40 0.574    0.05 11.48
N= 35  foundation prec 0.25 rec 0.30  depth 8
 N lesion  k   hit  chance  lift
35    row 10 0.148   0.029  5.19
35    row 40 0.111   0.029  3.89
35    col 10 0.546   0.029 19.12
35    col 40 0.389   0.029 13.61
N= 50  foundation prec 0.24 rec 0.31  depth 10
 N lesion  k   hit  chance  lift
50    row 10 0.102    0.02  5.09
50    row 40 0.130    0.02  6.48
50    col 10 0.222    0.02 11.11
50    col 40 0.333    0.02 16.67

total 484s

=== anchor recovery vs network size ===
N             11     20     35     50
lesion k                             
col    10  0.433  0.389  0

## Section 13 - Topological-order robustness

Corrupt the assumed order by adjacent transpositions and refit. Anchor accuracy drops
0.50 -> 0.17 **while edge precision/recall hold at 0.92/0.90 and no true edge becomes
unrepresentable**. Structural metrics do not predict adapter performance under order
misspecification.

In [21]:
# Section 13 - topological-order robustness (5 corruption draws per level)
def corrupt_mask(N, n_swap, seed=0):
    r = np.random.default_rng(seed); pos = np.arange(N)
    for _ in range(n_swap):
        i = int(r.integers(0, N-1)); pos[i], pos[i+1] = pos[i+1], pos[i]
    rank = np.empty(N, int); rank[pos] = np.arange(N)
    return (rank[:, None] < rank[None, :]).astype(np.float32)

class MaskSCM(FoundationSCM):
    def __init__(self, N, nc, depth, mask):
        super().__init__(N, nc, depth)
        with torch.no_grad():
            self.tri.copy_(torch.tensor(mask, device=self.tri.device))

rng0 = np.random.default_rng(0)
_specs = [{}] + [{int(x): KD_LEVEL} for x in rng0.choice(GVAL, 29, replace=False)]
ANC = [run_experiment(SIM, W_DIS, s, rng=rng0) for s in _specs]
TRUE = (SIM.W0 != 0)

rows_rb = []
for ns in [0, 120, 600]:
    for draw in range(5):
        M = corrupt_mask(GVAL, ns, seed=3 + draw)
        rev = float((TRUE & (M == 0)).sum() / TRUE.sum())
        f = fit_foundation(MaskSCM(GVAL, NCTX, SIM.depth, M).to(DEVICE), CORPUS, verbose=False)
        with torch.no_grad():
            Wh = f.W0().cpu().numpy()
        hs = np.abs(Wh) > 0.1; tp = (hs & TRUE).sum()
        GA = f.gains[DISEASE_CONTEXT]
        acc = [float((fit_ensemble(f, GA, ANC[:30], l1=ADAPT_L1, seed=s).anchors()
                      == FUSION["anchor"]).mean()) for s in range(3)]
        rows_rb.append({"swaps": ns, "pct_rev": round(rev*100, 1), "draw": draw,
                        "prec": round(float(tp/max(hs.sum(), 1)), 2),
                        "rec": round(float(tp/TRUE.sum()), 2),
                        "anchor_acc": round(float(np.mean(acc)), 3)})
        print(rows_rb[-1], flush=True)

rb = pd.DataFrame(rows_rb)
print()
print(rb.groupby("swaps")[["pct_rev", "prec", "rec", "anchor_acc"]]
        .agg(["mean", "std"]).round(3).to_string())
print(f"\nchance = {1/GVAL:.3f}")

{'swaps': 0, 'pct_rev': 0.0, 'draw': 0, 'prec': 0.92, 'rec': 0.91, 'anchor_acc': 0.5}
{'swaps': 0, 'pct_rev': 0.0, 'draw': 1, 'prec': 0.92, 'rec': 0.91, 'anchor_acc': 0.5}
{'swaps': 0, 'pct_rev': 0.0, 'draw': 2, 'prec': 0.92, 'rec': 0.91, 'anchor_acc': 0.5}
{'swaps': 0, 'pct_rev': 0.0, 'draw': 3, 'prec': 0.92, 'rec': 0.91, 'anchor_acc': 0.5}
{'swaps': 0, 'pct_rev': 0.0, 'draw': 4, 'prec': 0.92, 'rec': 0.91, 'anchor_acc': 0.5}
{'swaps': 120, 'pct_rev': 2.3, 'draw': 0, 'prec': 0.81, 'rec': 0.89, 'anchor_acc': 0.222}
{'swaps': 120, 'pct_rev': 1.1, 'draw': 1, 'prec': 0.87, 'rec': 0.9, 'anchor_acc': 0.278}
{'swaps': 120, 'pct_rev': 0.0, 'draw': 2, 'prec': 0.94, 'rec': 0.91, 'anchor_acc': 0.5}
{'swaps': 120, 'pct_rev': 2.3, 'draw': 3, 'prec': 0.91, 'rec': 0.89, 'anchor_acc': 0.222}
{'swaps': 120, 'pct_rev': 3.4, 'draw': 4, 'prec': 0.88, 'rec': 0.85, 'anchor_acc': 0.278}
{'swaps': 600, 'pct_rev': 5.7, 'draw': 0, 'prec': 0.8, 'rec': 0.85, 'anchor_acc': 0.5}
{'swaps': 600, 'pct_rev': 8.0, 'draw

## Reading the output

1. Foundation edge recovery > 0.8/0.8 (asserted, section 3).
2. Phenotype drop > 0.05 (asserted, section 1).
3. `anchor_hit` climbs with k - otherwise the interpretability claim goes.
4. `rank1` beats `dense` - otherwise the factorization is not load-bearing.
5. Either beats `no_adapt` - fails on `rescue_rho`; section 9 explains.
6. Active beats random.

## Limitations

Topological order is given, not discovered. All models are DAGs while signalling networks contain
feedback. The synthetic simulator shares a functional form with the model, only partly broken by
the semi-synthetic sections. `tanh` is a generic saturating nonlinearity, not Hill kinetics. The
robustness sweep uses three seeds and one lesion anchor - indicative, not tight.